# Chapter 00-01 · Start here: the course map, your notebook, and your first prediction

**Label:** Core  |  **Time:** ~50 minutes  |  **Difficulty:** gentle - no machine learning assumed

**Prerequisites:** you can read a Python `for` loop and call a function. That is genuinely all.

**Position in the learning path:** module 00 (Orientation), chapter 1 of 3. Nothing comes
before this. After this: **00-02**, which asks when machine learning is the wrong tool.

---

## Why this matters

Almost every machine learning disaster starts before any model is trained. Someone predicts
a thing that cannot be predicted at the moment it is needed, or measures success with a
number that does not match what a mistake actually costs, or celebrates a score that was
computed on the same data the model memorised.

None of that is fixed by knowing more algorithms. It is fixed by having a solid mental
model of what a prediction *is*, and by the habit of checking your rule against something
embarrassingly simple. This chapter builds both, on six rows of data you can check with a
pen.

## What you will be able to do

By the end of this chapter you can:

1. **State** what a prediction rule is, and what a row, a feature and a target are in a
   concrete table.
2. **Compute** mean absolute error by hand, with units, for a small table.
3. **Implement** a prediction rule and its error measure in a few lines of plain Python,
   and check that scikit-learn agrees with your arithmetic.
4. **Explain** why every model must be compared against a trivial baseline.
5. **Diagnose** the most common evaluation mistake in the field - scoring a rule on the
   data it was built from - and say why it makes a useless rule look perfect.

## First, three things about notebooks

This whole course lives in Jupyter notebooks. A notebook is a list of **cells**. A cell is
either text (like this one) or Python code.

1. **Run a cell with Shift+Enter.** Run them in order, from the top. The code cells share
   one memory, so a cell that uses `df` needs the cell that created `df` to have run first.
2. **`Kernel -> Restart Kernel and Run All Cells` is your undo button.** If anything ever
   behaves strangely, do that. Every notebook in this course is written to run correctly
   from a clean start; if one does not, that is a bug in the notebook, not in you.
3. **The last line of a code cell is displayed.** That is why cells often end with a bare
   variable name instead of `print(...)`.

Run the cell below now. It confirms your environment works. If it raises an error, go back
to [GETTING_STARTED.md](../../GETTING_STARTED.md) - do not push on, everything later
depends on this working.

In [ ]:
import sys

import matplotlib
import numpy as np
import pandas as pd

print("Python     ", sys.version.split()[0])
print("numpy      ", np.__version__)
print("pandas     ", pd.__version__)
print("matplotlib ", matplotlib.__version__)
print("\nIf you can read four version numbers above, you are ready.")

## The course in one paragraph

Fifteen modules. You start by learning to read data sceptically (module 02), then the small
amount of maths that is actually used (03), then the workflow that separates a working model
from an impressive-looking one (04). Only then do you meet models: regression (05),
classification (06), how to evaluate and explain them (07), learning without labels (08),
time series (09), neural networks (10), then text, images and recommenders (11), the rest of
the field honestly surveyed (12), and how to ship something you can defend and monitor (13).
Five capstone projects (14) make you decide things instead of following a recipe.

The full list, with prerequisites, is in [CURRICULUM.md](../../CURRICULUM.md).

Every chapter is built the same way: a human story, a table small enough to check by hand, a
prediction you make before running the code, a picture, one worked example with real
numbers, then the notation, then the library, then a realistic dataset, and finally a
**failure lab** where the method is broken on purpose so you can recognise the failure when
it happens quietly in your own work.

## Warm-up: what you already know

Later chapters open with retrieval questions from earlier chapters, because trying to
remember something is what makes it stick - far more than rereading it. This is chapter one,
so the questions come from your life instead.

Answer these out loud, from memory, before reading on:

1. Yesterday you guessed how long a journey would take. What information did you use?
2. Were you right? How did you decide whether you were "close enough"?
3. If you had to make that guess for a stranger's journey, what would you ask them first?

Hold on to your answers. You already own the three ideas this chapter formalises: the
**information you use**, the **thing you are guessing**, and **how wrong you were**.

## Where are you starting from?

A quick, honest self-check. For each statement, decide: *could I explain this to someone
else, out loud, right now?*

1. I can say what one row of a dataset represents in a specific project.
2. I can name the target and the features of a prediction problem.
3. I can compute mean absolute error by hand and state its units.
4. I can name a baseline for a prediction problem in under ten seconds.
5. I know why a model's score on its own training data is not evidence.
6. I can say when machine learning is the *wrong* tool for a task.
7. I can explain the difference between predicting something and explaining it.
8. I can name three ways information from the future can leak into a model.

**0-3 yes:** you are exactly the reader this course was written for. Start here, in order.

**4-6 yes:** read 00-01 quickly, but do not skip the failure lab, and do not skip module 04.

**7-8 yes:** skim module 00, take the diagnostic in **01-01** to decide about the Python
module, and start properly at **02-01**. Come back to any chapter whose failure lab surprises
you.

There is no score to record and nothing to submit. The only purpose is choosing where to
start.

---

## The situation

Maria rents bicycles from a stand at the edge of a park. Every morning she decides how many
bikes to bring out of the shed.

Bring too few and she turns customers away - lost income, and some of them do not come back.
Bring too many and she has paid a colleague to haul bikes that sat unused all day. She is not
looking for magic. She wants a rule she can follow at 7am with the information she has at
7am.

**The question this chapter answers:** what exactly *is* a rule like that, and how do we tell
a good one from a bad one?

### Six days from Maria's notebook

The table below is an **illustration**: six days I made up so that every number in this
chapter can be checked with a pen. It is not real data, and nothing about the real world
follows from it.

In [ ]:
import pandas as pd

days = pd.DataFrame({
    "day":     ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat"],
    "weather": ["sunny", "rainy", "sunny", "rainy", "sunny", "sunny"],
    "temp_c":  [22, 15, 25, 17, 19, 28],
    "rentals": [34, 12, 41, 15, 28, 50],
})
days

### What one row means

This matters more than it looks. Before anything else, in every project, answer this:

> **One row = one day at Maria's stand.**

Not one bike. Not one customer. Not one hour. If you get this wrong, everything downstream -
the split, the metric, the interpretation - is quietly wrong too.

Now the vocabulary, which you will use for the rest of the course:

| Term | In this table | The general idea |
|---|---|---|
| **Observation** (or row, or example) | one day | one thing you make a prediction about |
| **Target** (or label, or `y`) | `rentals` | the thing you want to know but will not know yet |
| **Feature** (or predictor, or `X`) | `weather`, `temp_c` | the information you *will* have when you predict |
| **Units** | rentals: bikes. temp: degrees Celsius | always know these - errors have units too |

Two things worth noticing already:

- `day` is a name, not information about the future. It is not a feature.
- `weather` and `temp_c` are only features **if Maria knows them at 7am**. Yesterday's
  observed weather is easy; tomorrow's weather is a forecast, which is a different and less
  reliable thing. This distinction destroys more real projects than any modelling mistake,
  and module 04 is largely about it.

**A prediction rule** is anything that turns the features of a row into a number for the
target. It can be a line of arithmetic, a lookup table, a decision tree, or a neural network
with a billion parameters. All of them are the same kind of object: information in, guess out.

### Predict before running

Write these down before running the next cell. Guessing and being wrong is the point.

1. Sunday will be **sunny, 24 °C**. What is your prediction for rentals? Write one number.
2. If Maria had to use the *same* number every single day - no features at all - what number
   would you tell her to use?
3. Rule A always says 30. Rule B says 38 on sunny days and 14 on rainy days. Which is wrong
   by less, on average, over these six days? By roughly how much?

In [ ]:
import matplotlib.pyplot as plt

colours = {"sunny": "#D55E00", "rainy": "#0072B2"}
markers = {"sunny": "o", "rainy": "s"}

fig, ax = plt.subplots(figsize=(6, 4))
for w, grp in days.groupby("weather"):
    ax.scatter(grp["temp_c"], grp["rentals"], s=90, label=w,
               color=colours[w], marker=markers[w])
ax.axhline(30, color="grey", linestyle="--", label="rule A: always 30")
ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Bikes rented (count)")
ax.set_title("Six made-up days at Maria's stand")
ax.legend()
plt.show()

## The intuition, in ordinary language

The dashed line is a rule. A very stubborn one: whatever the day looks like, it answers 30.

The dots are what actually happened. The **vertical gap** between a dot and the line is how
wrong the rule was on that day, in bikes. A better rule is one whose gaps are smaller.

That is the entire idea of supervised machine learning, and you now have it:

> Choose the rule whose gaps are small on days you have not looked at yet.

The last five words carry almost all the difficulty, and we will spend the rest of the
chapter - and much of module 04 - earning them.

**An analogy, and where it breaks.** People often say a model "learns like a student learns".
It is a useful picture for one thing only: practising on examples and being tested on new
ones. It breaks immediately afterwards. A student understands *why* an answer is right and
can tell you when a question is unfair or unanswerable. A fitted model has no notion of why;
it will answer confidently for a January day when it has only ever seen July, and it will
answer confidently for a question that makes no sense. When you catch yourself explaining a
model's behaviour by saying it "knows" or "understands" something, you have almost certainly
stopped describing what it does.

## The maths, slowly

**The question the formula answers:** "how wrong is this rule, in one number I can compare
between rules?"

### With real numbers first

Take rule A - always predict 30 bikes - and Monday, where 34 bikes were rented.

- actual: `34 bikes`
- predicted: `30 bikes`
- error: `34 - 30 = 4 bikes`

Do that for the first three days:

| Day | actual | predicted | error | size of error |
|---|---|---|---|---|
| Mon | 34 | 30 | +4 | 4 |
| Tue | 12 | 30 | -18 | 18 |
| Wed | 41 | 30 | +11 | 11 |

Why "size of error"? Because if we just add the errors, `+4` and `-18` partly cancel and a
rule that is wildly wrong in both directions looks fine. We want to punish being wrong, in
either direction. The simplest way is to drop the minus sign: the **absolute error**.

Average the three sizes:

`(4 + 18 + 11) / 3 = 33 / 3 = 11 bikes`

Over these three days, rule A is wrong by 11 bikes on average.

### Now the symbols

That average has a name, **mean absolute error**:

```
MAE = (1/n) * sum over i of | y_i - yhat_i |
```

- `n` - the number of rows you are scoring. Here 3, in a moment 6. A plain count, no units.
- `y_i` - the actual target for row `i`. Units: **bikes**.
- `yhat_i` ("y-hat") - what the rule predicted for row `i`. Units: **bikes**. The hat means
  "estimate of", and it is worth reading it that way every time you see it.
- `| ... |` - absolute value: drop the minus sign.
- The result is in **bikes**, the same unit as the target. That is MAE's best property: you
  can say the sentence "on average we are wrong by 11 bikes" to Maria and she can act on it.

**Back to English:** *take how wrong you were on each day, ignore the direction, and average.*

**Assumptions, and when this is the wrong thing to compute:** MAE treats a day that is 20
bikes off exactly twice as badly as a day 10 bikes off. Sometimes that is right. Sometimes
one huge failure is far worse than several small ones - then you want an error measure that
punishes big misses harder (chapter 05-04 introduces MSE and RMSE for exactly this). And
sometimes being 10 short costs more than being 10 over, because a turned-away customer never
returns; then neither is right and you need an asymmetric cost. **The metric is a claim about
what mistakes cost. Choose it deliberately.**

In [ ]:
# From scratch: the two ideas, in plain Python, no libraries.

def predict_constant(value, n_rows):
    """The simplest possible rule: ignore the features, always answer the same number."""
    return [value] * n_rows


def mean_absolute_error_by_hand(actual, predicted):
    """Average size of the gaps, in the units of the target."""
    sizes = [abs(a - p) for a, p in zip(actual, predicted)]
    return sum(sizes) / len(sizes)


first_three = [34, 12, 41]
print("sizes of the errors:", [abs(a - 30) for a in first_three])
print("MAE over 3 days:", mean_absolute_error_by_hand(first_three, predict_constant(30, 3)))

`11.0` - the same number we got with a pen. Now the whole table, and a second rule that
actually uses a feature.

Rule B says: *look at the weather, and answer with the average of the days that had that
weather.* The sunny days are 34, 41, 28, 50, which average to `153 / 4 = 38.25`. The rainy
days are 12 and 15, averaging `27 / 2 = 13.5`.

Notice what just happened: rule B is still only averages. It is not a clever rule. It just
takes one average and splits it into two, split by a piece of information.

In [ ]:
actual = days["rentals"].tolist()

overall_mean = sum(actual) / len(actual)
by_weather = days.groupby("weather")["rentals"].mean()

rule_a = predict_constant(overall_mean, len(actual))
rule_b = [by_weather[w] for w in days["weather"]]

print(f"overall mean          : {overall_mean:.2f} bikes")
print(f"mean by weather       : sunny {by_weather['sunny']:.2f}, rainy {by_weather['rainy']:.2f} bikes")
print(f"MAE rule A (always {overall_mean:.0f}) : {mean_absolute_error_by_hand(actual, rule_a):.2f} bikes")
print(f"MAE rule B (by weather) : {mean_absolute_error_by_hand(actual, rule_b):.2f} bikes")

### What just happened

Rule A is wrong by about **11.67 bikes** a day. Rule B is wrong by about **5.33 bikes** a day.
Adding a single piece of information - is it raining? - cut the average error by more than
half.

That is the whole promise of supervised learning, and it is worth stating plainly because
everything else is detail:

> **Using relevant information reduces error.** A model is a way of deciding *how* to use it.

Look at the picture again and you can see there is more information still on the table:
within the sunny days, the warmer ones are busier. Rule B ignores temperature entirely. A
rule that used it would do better still - that is chapter 05-02, and it will turn out to be
the same idea as here, just with a line instead of a lookup.

You can also check the hand arithmetic: for rule B the absolute errors are 4.25, 1.5, 2.75,
1.5, 10.25, 11.75, which sum to 32, and `32 / 6 = 5.33`. Nothing was hidden in the library.

In [ ]:
# The standard-library version. It should agree exactly with our arithmetic.
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

X = days[["temp_c"]]          # features, as a 2-D table: 6 rows, 1 column
y = days["rentals"]           # target, as a 1-D column

baseline = DummyRegressor(strategy="mean").fit(X, y)

print("sklearn baseline predicts:", baseline.predict(X)[:3], "...")
print("sklearn MAE  :", round(mean_absolute_error(y, baseline.predict(X)), 4))
print("our MAE      :", round(mean_absolute_error_by_hand(actual, rule_a), 4))

### What scikit-learn is, and is not

scikit-learn is the standard Python library for classical machine learning. Nearly every
model in it follows the same two-step shape, which you have now seen once and will see two
hundred times:

- **`.fit(X, y)`** - look at training data and set the rule's internals. Here: compute one
  average.
- **`.predict(X)`** - apply the rule to rows and return predictions.

`DummyRegressor` is deliberately stupid: it ignores `X` completely. It exists so you always
have something to beat. If your careful model cannot beat `DummyRegressor`, your model has
found nothing, no matter how sophisticated it is.

And note what the check above proves: the library did the same arithmetic you did by hand.
Calling a library is not understanding, and understanding does not require avoiding
libraries. Throughout this course you will build the mechanism once, small, then use the
library version - which is faster, tested, and handles edge cases you have not thought of.

---

## On more data than you can check by hand

Six rows are perfect for arithmetic and useless for judging a rule. So here are 60 days.

They are **SYNTHETIC** - generated by the code below, not measured anywhere. That is a
deliberate choice, not a shortcut: because we wrote the generator, we *know* the true
relationship, so we can check whether our methods find it. Real data never offers that. The
rule for the rest of the course is absolute: **never quote a number from synthetic data as a
fact about the world.**

What we built in on purpose: warmer days are busier, rain costs about 22 rentals, and there
is day-to-day randomness nothing could predict. Counts are rounded to whole bikes and floored
at zero, so a cold rainy day can read 0 - that floor is a real property of counts, and it is
the kind of detail that quietly breaks methods which assume any value is possible.

In [ ]:
import numpy as np

rng = np.random.default_rng(1)          # fixed seed: you get exactly these 60 days too

n_days = 60
temp_c = rng.integers(10, 29, size=n_days)                 # whole degrees Celsius
is_rainy = rng.random(n_days) < 0.40
rentals = 1.5 * temp_c - 22 * is_rainy + rng.normal(0, 3, n_days)

season = pd.DataFrame({
    "day": np.arange(1, n_days + 1),
    "temp_c": temp_c,
    "weather": np.where(is_rainy, "rainy", "sunny"),
    "rentals": np.clip(np.round(rentals), 0, None).astype(int),   # SYNTHETIC
})
season.head()

### Days we have seen, and days we have not

Here is the first genuinely important habit in this course.

Maria's rule has to work on **tomorrow**, and tomorrow is a day nobody has seen. So we hide
some days from ourselves: build the rule using days 1-40 only, then judge it on days 41-60,
which the rule never saw.

The hidden days are called a **held-out set** (or test set). Days used to build the rule are
the **training set**. Module 04 makes this rigorous - how big, how to split, when a random
split is a lie. For now, one sentence is enough:

> A score computed on days used to build the rule is not evidence. A score on days the rule
> has never seen is.

In [ ]:
train = season.iloc[:40]      # days 1-40: we may look at these
test = season.iloc[40:]       # days 41-60: hidden until we score

train_mean = train["rentals"].mean()
train_by_weather = train.groupby("weather")["rentals"].mean()

def score(rows, predictions, name):
    mae = mean_absolute_error(rows["rentals"], predictions)
    print(f"{name:<28} MAE = {mae:5.2f} bikes")

print("Judged on the 20 held-out days:")
score(test, [train_mean] * len(test), "rule A: always the average")
score(test, test["weather"].map(train_by_weather), "rule B: average per weather")

Rule B wins again, and by a wide margin - about 5 bikes a day better. Two details in that
code are worth pausing on, because they are habits rather than syntax:

- `train_mean` and `train_by_weather` are computed from **training days only**. Using all 60
  days to compute the average, then scoring on 20 of them, would quietly let the answer leak
  into the rule. This is the smallest possible example of **leakage**, the subject of 04-05.
- Rule A is the **baseline**. It is not a straw man to make rule B look good; it is the
  honest question *"how much did this idea actually buy me?"*. Ask it of every model you
  ever build. A baseline is also a safety net: sometimes the sophisticated model loses, and
  you want to find that out yourself rather than in a review meeting.

In [ ]:
# Diagnostics: one number hides more than it tells. Look at the error by segment.
test_pred = test["weather"].map(train_by_weather)
errors = pd.DataFrame({
    "weather": test["weather"],
    "actual": test["rentals"],
    "predicted": test_pred.round(1),
    "abs_error": (test["rentals"] - test_pred).abs().round(1),
})
errors.groupby("weather")[["abs_error"]].agg(["count", "mean", "max"]).round(2)

### Reading a diagnostic

Look at the two rows rather than the single MAE of 7.51.

The rule is clearly worse on sunny days (about 8.7 bikes off) than on rainy ones (about 5.8),
and its worst single day is 14.6 bikes off - roughly double the headline number, which hid it
completely.

This is the reflex to build now, and every chapter from here will reinforce it:

> **Never accept a single number as a description of a model.** Ask: on which rows is it
> wrong, by how much, and does that pattern make sense?

Here the pattern makes sense, and it points somewhere useful. Within each weather group the
rule answers one constant, so it cannot tell a 10 °C day from a 28 °C one. Sunny days spread
across a wider range of demand than rainy ones, so one constant hurts more there. The
leftover error is temperature that we are refusing to use - exactly the gap module 05 fills.

---

## Failure lab: the rule that scores perfectly and is worth nothing

Time to break something on purpose.

Here is a rule with a wonderful score. It works by looking up the day number and answering
with exactly what happened on that day. For any day it has seen, it is never wrong. Watch
what it does on the training days - then on the held-out ones.

**Predict before running:** what MAE will this rule get on days 1-40? And on days 41-60?

In [ ]:
# A rule that memorises: day number -> exactly what happened that day.
lookup = dict(zip(train["day"], train["rentals"]))
fallback = train_mean          # for a day it has never seen, it has nothing to say

def memoriser(day_numbers):
    return [lookup.get(d, fallback) for d in day_numbers]

print("Judged on the 40 days it was BUILT from:")
score(train, memoriser(train["day"]), "rule C: memorise the day")
print("\nJudged on the 20 days it has never seen:")
score(test, memoriser(test["day"]), "rule C: memorise the day")
score(test, [train_mean] * len(test), "rule A: always the average")

### Diagnosis

**MAE 0.00 on the training days.** A flawless score. Report that number without saying where
it came from and it sounds like a breakthrough.

**On new days it is exactly as good as answering with the average** - because for a day it
has never seen, answering with the average is literally all it does. It learned nothing about
*why* busy days are busy. It stored answers.

Two lessons, and the second is the one that matters:

1. A score on the data a rule was built from measures **memory**, not skill. It is not weak
   evidence; it is no evidence.
2. You object, correctly, that rule C is obviously stupid. It is - and that is the trap.
   A model with a million parameters can do precisely this while looking sophisticated: fit
   the training rows beautifully, generalise to nothing. The failure is identical, only the
   silliness is hidden. It has a name, **overfitting**, and chapters 05-07 and 05-08 take it
   apart properly. The reason you can already recognise it is that you have now seen the
   naked version.

### Remedies, and what each costs

| Remedy | What it fixes | What it costs | When to prefer it |
|---|---|---|---|
| Hold out days and score there | Stops memory being mistaken for skill | You lose those days for building | Always. This is not optional |
| Compare against a baseline | Tells you what the model actually bought | A few lines of code | Always |
| Look at error by segment | Reveals failures the average hides | A little thought | Whenever the decision matters |
| Use fewer, more meaningful features | Reduces what there is to memorise | Might discard real signal | Covered in 05-09 |

### The second trap, which you can now see coming

Rule C failed by using information it would not have. Suppose instead we build a rule that
uses **the actual weather of the day being predicted**. That works beautifully in this
notebook. But at 7am Maria does not have today's actual weather - she has a **forecast**,
which is sometimes wrong.

So a rule that looks excellent here can be worse in practice, not because the model is bad
but because we evaluated it with information that will not exist at the moment of the
decision. That is **leakage**, and it is the single most expensive mistake in applied machine
learning. Module 04 is largely about hunting it down. For now, carry one question with you:

> **Would I actually have this number at the moment I need to make the prediction?**

## Common misconceptions

**"A model finds the true cause of things."**
No. Rule B found that rainy days are quieter. It has no idea whether rain drives people away
or whether Maria simply opens later when it rains. Prediction and causation are different
questions with different methods; 00-03 separates them and 12-07 goes deep.

**"More features always help."**
Rule C had a feature - the day number - and was useless. What helps is *relevant information
available at prediction time*. A feature that is neither is not neutral; it gives the model
something to memorise.

**"99% accurate means the model is good."**
It means nothing until you know what a mistake costs, what a trivial baseline scores, and
which rows it fails on. If 99% of days are ordinary, "always predict ordinary" is also 99%
right and is worthless. Module 06 makes this concrete.

**"The model was 95% right in testing, so it will be 95% right in production."**
Only if tomorrow resembles the days you tested on. Weather changes, prices change, the world
changes. This is **drift**, and 13-08 is about watching for it.

**"Simple methods are for teaching; real work uses deep learning."**
Rule B is two averages and it halved the error. In a great many real problems, a well-framed
simple model with good features beats a poorly-framed complex one - and it can be explained,
debugged and deployed. Complexity is a cost you pay for a benefit you must demonstrate.

---

## Exercises

Work these before opening the solutions. The solutions explain the *reasoning*, not just the
code, so read them even when you got the answer right:
`solutions/00_orientation/00-01_start_here_solutions.ipynb`.

Use the empty cell at the end for the coding ones.

### Quick understanding

**E1 (define).** In your own words, in one sentence each: what is a prediction rule, a
feature, and a target?

**E2 (explain).** In Maria's six-day table, what does one row represent? Name the target,
name two features, and name one column that is *not* a feature. Say why.

**E3 (explain).** Why is it worth computing "always predict the average" even when you are
certain you will build something better?

### Hand calculation

**E4 (calculate).** With a pen, compute the MAE of the rule "always predict 35" on the six
days (34, 12, 41, 15, 28, 50). Show each absolute error, then the average. Is 35 better or
worse than 30, and by how many bikes?

**E5 (calculate).** The six-day mean is 30 and the median is 31. A seventh day is added:
sunny, 30 °C, **120 rentals**, because a festival came to the park. Recompute both the mean
and the median. Which moved more? What does that suggest about which summary you would trust
as a baseline when a dataset has rare enormous days?

### Coding

**E6 (calculate + code).** Using `mean_absolute_error_by_hand`, score the constant rules 25,
30, 35 and 40 on the six days. Then try every whole number from 0 to 60 and plot MAE against
the constant. Which constant is best? Is it the mean, the median, or neither? (You are
discovering something real about MAE here - 05-01 explains it.)

**E7 (design + code).** Add a third rule to the 60-day experiment: **"predict the average of
the previous 5 days"**. Build it so that predicting day `d` uses only days `d-5` to `d-1`.
Score it on days 41-60 and compare with rules A and B. Does it win? Why might a rule like this
be attractive to Maria even if it does not?

### Interpretation

**E8 (interpret).** Maria looks at "MAE = 7.5 bikes" and says: *"that's terrible."* Give two
specific pieces of information you would need before you could agree or disagree with her.

### Debugging

**E9 (diagnose).** A colleague messages you: *"Great news, my model gets MAE 0.4 bikes."* List
the three questions you would ask, in order, and say what a worrying answer to each would
sound like.

### Exam and interview reasoning

**E10 (defend).** *"Why not just always use the most complex model available?"* Answer in
four sentences, using rule C as your evidence.

**E11 (design).** An interviewer says: *"You have a year of hourly bike rental data for a
city, and you want to predict demand for tomorrow. What is the first model you build?"*
Answer in three sentences, and state one thing you would check before building anything.

### Transfer to a different situation

**E12 (design).** A hospital wants to predict how many beds will be needed tomorrow. Write
five short lines: (a) what one row is, (b) what the target is and its units, (c) one baseline
a nurse could compute in their head, (d) which error measure and why, (e) one piece of
information that will *not* be available at 6pm today when the prediction is needed.

### Explain it to someone non-technical

**E13 (explain).** In at most 60 words and with no numbers, explain to Maria why judging a
rule on the same days it was built from tells you nothing. Use one everyday analogy - and
then add a sentence saying where your analogy stops being accurate.

### Optional challenge

**E14 (design + code).** Build a rule that uses temperature: to predict a new day, average
the rentals of the **three training days with the closest temperature**. Score it on days
41-60 against rules A and B. You have just implemented k-nearest neighbours by hand - you
will meet it formally in 06-10.

In [ ]:
# Your workspace for the coding exercises.
# Everything from the chapter is still in memory: days, season, train, test,
# mean_absolute_error_by_hand, predict_constant, train_mean, train_by_weather.

## Mastery check

Without scrolling up, can you:

- [ ] Say what one row represents, and name the target and features, for Maria's table?
      *(If not: "What one row means".)*
- [ ] Compute MAE for three rows on paper, and state its units? *(If not: "The maths, slowly".)*
- [ ] Say in one sentence why a baseline is compulsory rather than optional?
      *(If not: "Days we have seen, and days we have not".)*
- [ ] Explain why rule C got a perfect score and was still worthless? *(If not: "Failure lab".)*
- [ ] Name the question you must ask of every feature before using it? *(If not: "The second
      trap".)*

## What should now feel instinctive

When you meet any prediction problem from now on, these should happen before you think about
models at all:

1. **"What is one row?"** - asked first, every time, and answered in words.
2. **"What is the target, in what units, and when do I need the answer?"**
3. **"What is the stupidest rule that could work?"** - and then actually computing its score.
4. **"Am I scoring this on data the rule has already seen?"** - a suspiciously good number is
   a symptom, not a success.
5. **"Would I really have this information at prediction time?"** - asked of every single
   feature.

## Flashcards

| Question | Answer |
|---|---|
| What is a prediction rule? | Anything that turns a row's features into a guess about its target |
| What is an observation? | One thing you make a prediction about - one row |
| Target vs feature? | Target: what you want to know. Features: what you will know when you predict |
| Define MAE in one sentence | The average size of the errors, ignoring their direction, in the units of the target |
| Units of MAE if the target is bikes? | Bikes |
| Why absolute values in MAE? | So errors in opposite directions cannot cancel out and hide badness |
| What is a baseline? | A trivially simple rule your model must beat before it is worth anything |
| What does `.fit` do, what does `.predict` do? | `fit`: set the rule's internals from training data. `predict`: apply the rule to rows |
| Why is a training-set score not evidence? | It measures memory of seen rows, not skill on unseen ones |
| One sentence definition of overfitting | Fitting the training rows so closely that the rule stops working on new ones |
| The one question to ask of every feature | Would I actually have this value at the moment I make the prediction? |

## Next

**Chapter 00-02 · What machine learning is, what it is not, and when a rule wins.**

You now know that a model is a rule chosen from data. The obvious next question is when
choosing a rule from data is a *bad* idea - and it often is. Sometimes an `if` statement, a
database query, a designed experiment or a phone call to a domain expert is faster, cheaper,
more accurate and easier to defend. Being able to say "this does not need machine learning"
is one of the most valuable things you will learn here, and it comes next.

New terms from this chapter are collected in [GLOSSARY.md](../../GLOSSARY.md).